In [ ]:
 # this notebook conduct the following task:
# 1 Extract the positive patch information from the data frame, and then convert patch info into 15*20 matrix
#   /content/drive/My Drive/@antibody/antigen_epitope/abdb_v2/data_processed_step1_with_SI/
# 2 Use other .ipynb files to process the negative dataset

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v3/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/abdb_v2/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd
import itertools
import math

# Base path for the folds
base_fold_path = '/content/drive/MyDrive/@antigen_epitope/abdb_v4'

In [ ]:
#@title Extract the positive patch information from the data frame, and then convert patch info into 15*20 matrix
# see TEST for more details

# One-hot encoding for amino acid types
def encode_aa(aa):
    aa_types = ['ALA', 'CYS', 'ASP', 'GLU', 'PHE', 'GLY', 'HIS', 'ILE', 'LYS', 'LEU',
                'MET', 'ASN', 'PRO', 'GLN', 'ARG', 'SER', 'THR', 'VAL', 'TRP', 'TYR']
    return [1 if aa_type == aa else 0 for aa_type in aa_types]

# One-hot encoding for secondary structure
def encode_ss(ss):
    return [1 if ss == 'S' else 0, 1 if ss == 'L' else 0, 1 if ss == 'H' else 0]

# One-hot encoding for SaSaRate (Now 5 levels)
def encode_sasa(sasa):
    if sasa < 0.2:
        return [1, 0, 0, 0, 0]
    elif sasa < 0.4:
        return [0, 1, 0, 0, 0]
    elif sasa < 0.6:
        return [0, 0, 1, 0, 0]
    elif sasa < 0.8:
        return [0, 0, 0, 1, 0]
    else:
        return [0, 0, 0, 0, 1]

# Combine 3-d one-hot encoded vector for secondary structure with 5-d one-hot encoded vector for SaSaRate
# Resulting in a 15-d vector
def combine_ss_sasa(ss_encoded, sasa_encoded):
    combined = []
    for i in ss_encoded:
        for j in sasa_encoded:
            combined.append(i * j)
    return combined  # This should be a 15-d vector

def calculate_distance(residue1, residue2):
    x1, y1, z1 = residue1['SCx'], residue1['SCy'], residue1['SCz']
    x2, y2, z2 = residue2['SCx'], residue2['SCy'], residue2['SCz']
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)

# Process each fold's training set
for fold in range(1, 6):

    # Specify the directory and filename of the combined data CSV
    dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data')
    file_name = 'all_data_with_SI.csv'

    # Read the CSV file without setting the first column as an index
    combined_df = pd.read_csv(os.path.join(dir_name, file_name))
    filtered_combined_df = combined_df

    save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4' + f'/fold_{fold}/data_processed_step1_with_SI/'
    os.makedirs(save_dir, exist_ok=True)

    dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4' + f'/fold_{fold}/data_processed_step1_with_SI/cnn_positive_patch_3points.txt'

    # Create or overwrite cnn_positive_patch_3points.txt at the beginning
    with open(dir, 'w') as f:
        f.write('')  # Writing an empty string to ensure the file is empty

    # Initialize an empty list to store all positive patches
    positive_patches = []

    # Loop through each group (protein)
    for name, group in filtered_combined_df.groupby('name'):
        # Filter residues with SaSaRate > 0.1
        filtered_group = group[group['SaSaRate'] > 0.1]

        # Filter interacting residues with SaSaRate > 0.1
        interacting_residues = filtered_group[filtered_group['Status'] == 'Interacting']
        patches = []

        # Loop over each interacting residue
        for i, (index1, row1) in enumerate(interacting_residues.iterrows()):
            for j in range(i + 1, len(filtered_group)):
                row2 = filtered_group.iloc[j]
                d12 = calculate_distance(row1, row2)

                if d12 < 8.5:
                    for k in range(j + 1, len(filtered_group)):
                        row3 = filtered_group.iloc[k]
                        d23 = calculate_distance(row2, row3)
                        d31 = calculate_distance(row3, row1)

                        if d23 < 8.5 and d31 < 8.5:
                            patch_indices = sorted([row1['ResID'], row2['ResID'], row3['ResID']])
                            patch_info = [
                                [row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                                [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                                [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]
                            ]

                            if patch_indices not in [p['indices'] for p in patches]:
                                patches.append({'indices': patch_indices, 'info': patch_info})

        # Output patches to a text file
        with open(dir, 'a') as f:
            f.write(f"Patches for protein {name}:\n")
            for patch in patches:
                f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

        # Convert each patch into a 15*20 matrix and append it to positive_patches
        for patch in patches:
            patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
            for idx, aa_info in enumerate(patch['info']):
                aa = aa_info[0]
                ss = aa_info[1]
                sasa = aa_info[2]
                si = aa_info[3]  # Extract the SI value

                aa_encoded = encode_aa(aa)
                ss_encoded = encode_ss(ss)
                sasa_encoded = encode_sasa(sasa)

                combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

                for i in range(20):
                    if aa_encoded[i] == 1:
                        patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value

            positive_patches.append(patch_matrix)

    # Convert positive_patches to a NumPy array
    positive_patches_3points = np.array(positive_patches)

    # Save the NumPy array
    np.save(f'{save_dir}cnn_positive_patch_3points.npy', positive_patches_3points)


In [ ]:
for fold in range(1, 6):

    # Specify the directory and filename of the combined data CSV
    dir_name = os.path.join(base_fold_path, f'fold_{fold}', 'data')
    file_name = 'all_data_with_SI.csv'

    # Read the CSV file without setting the first column as an index
    combined_df = pd.read_csv(os.path.join(dir_name, file_name))
    filtered_combined_df = combined_df

    save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4' + f'/fold_{fold}/data_processed_step1_with_SI/'
    os.makedirs(save_dir, exist_ok=True)

    dir = '/content/drive/MyDrive/@antigen_epitope/abdb_v4' + f'/fold_{fold}/data_processed_step1_with_SI/cnn_positive_patch_3points.txt'

    # Create or overwrite cnn_positive_patch_3points.txt at the beginning
    with open(dir, 'w') as f:
        f.write('')  # Writing an empty string to ensure the file is empty

    # Initialize an empty list to store all positive patches
    positive_patches = []

    # Loop through each group (protein)
    for name, group in filtered_combined_df.groupby('name'):
        #interacting_residues = group[group['Status'] == 'Interacting']
        interacting_residues = group[(group['Status'] == 'Interacting') & (group['SaSaRate'] > 0.1)]
        patches = []

        for comb in itertools.combinations(interacting_residues.iterrows(), 3):
            index1, row1 = comb[0]
            index2, row2 = comb[1]
            index3, row3 = comb[2]

            d12 = calculate_distance(row1, row2)
            d23 = calculate_distance(row2, row3)
            d31 = calculate_distance(row3, row1)

            if d12 < 8.5 and d23 < 8.5 and d31 < 8.5:
                patch_indices = [row1['ResID'], row2['ResID'], row3['ResID']]

                patch_info = [[row1['AA'], row1['SS'], row1['SaSaRate'], row1['SI']],
                              [row2['AA'], row2['SS'], row2['SaSaRate'], row2['SI']],
                              [row3['AA'], row3['SS'], row3['SaSaRate'], row3['SI']]]

                patches.append({'indices': patch_indices, 'info': patch_info})

        # Output patches to a text file
        with open(dir, 'a') as f:
            f.write(f"Patches for protein {name}:\n")
            for patch in patches:
                f.write(f"indices: {patch['indices']}, info: {patch['info']}\n")

        # Convert each patch into a 15*20 matrix and append it to positive_patches
        for patch in patches:
            patch_matrix = np.zeros((15, 20))  # Initialize a 15x20 matrix with zeros
            for idx, aa_info in enumerate(patch['info']):
                aa = aa_info[0]
                ss = aa_info[1]
                sasa = aa_info[2]
                si = aa_info[3]  # Extract the SI value

                aa_encoded = encode_aa(aa)
                ss_encoded = encode_ss(ss)
                sasa_encoded = encode_sasa(sasa)

                combined_info = combine_ss_sasa(ss_encoded, sasa_encoded)  # 15-d vector

                for i in range(20):
                    if aa_encoded[i] == 1:
                        patch_matrix[:, i] += np.array(combined_info) * si  # Multiply the SI value

            positive_patches.append(patch_matrix)

    # Convert positive_patches to a NumPy array
    positive_patches_3points = np.array(positive_patches)

    # Save the NumPy array
    np.save(f'{save_dir}cnn_positive_patch_3points.npy', positive_patches_3points)